# Módulo 00 — internals do modelo: a base cheia

O [walkthrough](srag_model_walkthrough.ipynb) treina na amostra commitada.
Este caderno paga a conta completa — **onde a base cheia existe**: o Ouro
em disco (`~/Documents/srag-data/gold/`). Reconstruí-lo do Bronze estoura a
memória de um runner de CI (o `S.build` materializa 1,7 M × 420 no pior
ano — OOM medido em 2026-09-01, o modo de falha que o próprio módulo 00
documenta). Em ambiente sem o Ouro, cada seção pesada **imprime o pulo
declarado** em vez de fingir; os outputs commitados vêm de uma execução
local completa, e é isso que a barra de evidência exige.

In [1]:
%pip install -q pandas pyarrow scikit-learn xgboost

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pathlib
import subprocess
import sys
import time

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

REPO = pathlib.Path.cwd()
while REPO != REPO.parent and not (REPO / "tools" / "srag_model.py").exists():
    REPO = REPO.parent
if not (REPO / "tools" / "srag_model.py").exists():
    import tempfile
    REPO = pathlib.Path(tempfile.gettempdir()) / "interpretable-ml-lectures"
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/wbendinelli/interpretable-ml-lectures.git",
                        str(REPO)], check=True)
sys.path.insert(0, str(REPO / "tools"))
import srag_model as M

OURO_PATH = pathlib.Path.home() / "Documents/srag-data/gold/gold_covid_obito.parquet"
if OURO_PATH.exists():
    ouro = pd.read_parquet(OURO_PATH)
    for c in M.CATEGORICAS:
        ouro[c] = ouro[c].astype("category")
    print(f"Ouro: {len(ouro):,} × {len(ouro.columns)}")
    print(ouro.groupby("split", observed=True)["y_obito"]
          .agg(["size", "sum", "mean"]).round(3).to_string())
else:
    ouro = None
    print("SEM Ouro local — as seções de base cheia imprimem o pulo declarado;")
    print("os outputs commitados vêm de uma execução local completa.")

Ouro: 1,282,970 × 50
          size     sum   mean
split                        
test     16142    2945  0.182
train  1242680  390769  0.314
val      24148    4572  0.189


## 1. O preço da amostra: fit cheio vs fit de 200 mil

O delta é o número que autoriza o desenho do curso: se for pequeno,
explicar o modelo-da-amostra é explicar o modelo. A avaliação é idêntica
nos dois (val e teste inteiros, por construção da amostra).

In [3]:
amostra = M.load_gold(REPO / "modules/00-dataset/gold/gold_covid_obito_sample.parquet")
t0 = time.time()
xgb_amostra, _ = M.fit_models(amostra)
t_amostra = time.time() - t0

pares = [("amostra (200 k)", xgb_amostra, amostra, t_amostra)]
xgb_cheio = None
if ouro is not None:
    t0 = time.time()
    xgb_cheio, _logit_cheio = M.fit_models(ouro)
    pares.insert(0, ("cheio (1,24 M)", xgb_cheio, ouro, time.time() - t0))
else:
    print("PULADO (fit cheio): sem Ouro local.")

linhas = []
for nome, modelo, base, t in pares:
    for s in ("val", "test"):
        linhas.append({"treino": nome, "fit_s": round(t, 1)} | M.metrics(modelo, base, s))
print(pd.DataFrame(linhas).to_string(index=False))

         treino  fit_s split     n  obitos    auc  brier  previsto_medio  observado
 cheio (1,24 M)   23.9   val 24148    4572 0.7601 0.1355          0.2253     0.1893
 cheio (1,24 M)   23.9  test 16142    2945 0.7680 0.1304          0.2196     0.1824
amostra (200 k)    4.9   val 24148    4572 0.7564 0.1356          0.2209     0.1893
amostra (200 k)    4.9  test 16142    2945 0.7644 0.1308          0.2154     0.1824


## 2. AUC por ano de início — a deriva, medida onde dói

In [4]:
from sklearn.metrics import roc_auc_score

if ouro is None:
    print("PULADO: exige a base cheia (outputs commitados carregam a tabela).")
else:
    x_xgb, _, y = M.design_matrix(ouro)
    p = xgb_cheio.predict_proba(x_xgb)[:, 1]
    por_ano = []
    for ano in sorted(ouro["ano_onset"].unique()):
        m = (ouro["ano_onset"] == ano).to_numpy()
        if ouro.loc[m, "y_obito"].nunique() < 2:
            continue
        por_ano.append({
            "ano": int(ano), "n": int(m.sum()),
            "CFR %": round(100 * float(ouro.loc[m, "y_obito"].mean()), 1),
            "AUC": round(float(roc_auc_score(y[m], p[m])), 4),
            "previsto médio": round(float(p[m].mean()), 3),
        })
    print(pd.DataFrame(por_ano).to_string(index=False))

 ano      n  CFR %    AUC  previsto médio
2020 407679   32.2 0.7890           0.322
2021 723037   31.4 0.7546           0.314
2022 111964   28.8 0.7451           0.289
2023  24148   18.9 0.7601           0.225
2024  16142   18.2 0.7680           0.220


A discriminação (AUC) resiste melhor que a calibração: o ranking de risco
viaja entre regimes; o nível absoluto, não.

## 3. Curva de aprendizado — quanto treino compra o quê

In [5]:
if ouro is None:
    print("PULADO: exige a base cheia.")
else:
    rng = np.random.default_rng(42)
    treino_cheio = ouro[ouro["split"] == "train"].reset_index(drop=True)
    aval = ouro[ouro["split"] != "train"]
    curva = []
    for n_treino in (50_000, 200_000, len(treino_cheio)):
        if n_treino < len(treino_cheio):
            idx = np.sort(rng.choice(len(treino_cheio), size=n_treino, replace=False))
            base = pd.concat([treino_cheio.iloc[idx], aval], ignore_index=True)
        else:
            base = ouro
        for c in M.CATEGORICAS:
            base[c] = base[c].astype("category")
        mdl, _ = M.fit_models(base)
        curva.append({"treino": n_treino} | {
            k: v for k, v in M.metrics(mdl, base, "test").items()
            if k in ("auc", "brier")
        })
    print(pd.DataFrame(curva).to_string(index=False))

 treino    auc  brier
  50000 0.7484 0.1331
 200000 0.7656 0.1307
1242680 0.7680 0.1304


## 4. O exemplar sobrevive ao modelo cheio?

In [6]:
ex_amostra = M.pick_exemplar(xgb_amostra, amostra)
print(f"regra na amostra   → gold_id {int(ex_amostra.gold_id)}  p = {ex_amostra.p_obito:.3f}")
if ouro is None:
    print("PULADO (comparação com o modelo cheio): sem Ouro local.")
else:
    ex_cheio = M.pick_exemplar(xgb_cheio, ouro)
    print(f"regra na base cheia → gold_id {int(ex_cheio.gold_id)}  p = {ex_cheio.p_obito:.3f}")
    p_do_exemplar = float(M.predict_proba(
        xgb_cheio, ouro[ouro["gold_id"] == int(ex_amostra.gold_id)]
    )[0])
    print(f"p do exemplar-da-amostra sob o modelo cheio: {p_do_exemplar:.3f}")
print("\nO paciente do curso é o da AMOSTRA — o objeto de estudo dos módulos.")
print("Se as regras divergem entre amostra e base, é registro, não problema.")

regra na amostra   → gold_id 1276776  p = 0.500


regra na base cheia → gold_id 1270631  p = 0.500
p do exemplar-da-amostra sob o modelo cheio: 0.441

O paciente do curso é o da AMOSTRA — o objeto de estudo dos módulos.
Se as regras divergem entre amostra e base, é registro, não problema.


## 5. As quatro impossibilidades: amostra vs base cheia

A regra do repositório: medir no cheio E re-medir na amostra, imprimir os
dois — diferença além de composição é achado sobre a amostra.

In [7]:
def fracoes(base):
    n = len(base)
    return {
        "portão do funil": 100 * int((~base["fator_risc_portao"].astype(bool)).sum()) / n,
        "pré-campanha": 100 * int((base["meses_desde_mar2020"] < M.MESES_CAMPANHA).sum()) / n,
        "critério-2 por um fio": 100 * int((base["n_crit2"] == 1).sum()) / n,
        "critério-3 por um fio": 100 * int((base["n_crit3"] == 1).sum()) / n,
    }

colunas = {"amostra %": fracoes(amostra)}
if ouro is not None:
    colunas = {"base cheia %": fracoes(ouro)} | colunas
quadro = pd.DataFrame(colunas).round(2)
if "base cheia %" in quadro:
    quadro["Δ"] = (quadro["amostra %"] - quadro["base cheia %"]).round(2)
print(quadro.to_string())
print("\nA amostra sub-pesa o pré-campanha porque val/test (2023-24, inteiros)")
print("pesam mais nela — efeito de composição declarado; os módulos citam a")
print("fração da base em que operam.")

                       base cheia %  amostra %     Δ
portão do funil               38.05      37.27 -0.78
pré-campanha                  34.50      29.67 -4.83
critério-2 por um fio         78.83      79.60  0.77
critério-3 por um fio         23.66      24.67  1.01

A amostra sub-pesa o pré-campanha porque val/test (2023-24, inteiros)
pesam mais nela — efeito de composição declarado; os módulos citam a
fração da base em que operam.


## 6. Por que `UTI` e `SUPORT_VEN` ficam fora: medido, não afirmado

As duas colunas clinicamente mais fortes da ficha estão na classe
`leakage` ([GOLD.md](../GOLD.md), decisão 3), e a pergunta volta sempre:
se predizem tanto, por que não usá-las? Porque a ficha pergunta se o
paciente **fez uso** de suporte ventilatório e se **foi internado** em
UTI — do episódio inteiro, sem data — e só é encerrada depois que a
evolução é digitada. O campo nasce no mesmo ato que preenche o rótulo:
não é medida da admissão, é resumo do que aconteceu depois. Usá-lo é
mover o instante da predição para depois do desfecho, o que a pergunta
2.3 do PROBAST chama de preditor indisponível no momento de uso.

Afirmar isso é barato; esta seção mede. As colunas não existem no Ouro
(o funil as descarta), então voltam do **Prata**, pela MESMA função de
coorte que construiu o Ouro, realinhadas por `gold_id` — e o realinhamento
é conferido antes de qualquer número. Sem o Prata local, a seção imprime
o pulo declarado; os outputs commitados vêm de uma execução local.

In [8]:
import pyarrow.parquet as pq

import srag_gold as G

PRATA = pathlib.Path.home() / "Documents/srag-data/silver.parquet"
MAPA_SV = {"1": "invasivo", "2": "nao_invasivo", "3": "nao", "9": "ignorado"}
MAPA_UTI = {"1": "sim", "2": "nao", "9": "ignorado"}

com_uti = None
if not PRATA.exists():
    print("PULADO: exige o Prata local (~/Documents/srag-data/silver.parquet);")
    print("os outputs commitados vêm de uma execução local completa.")
else:
    decisoes = G.Decisions(  # as decisões commitadas em gold/counts.json
        coorte="hospitalizado",
        etiologia="covid-amplo",
        alvo="obito-casos-fechados",
        inicio=pd.Timestamp("2020-02-26"),
        idade="todas",
        idade_maxima=120.0,
        idade_ausente="excluir",
        nosocomial="manter",
        split="temporal:2022-12-31/2023/2024",
        amostra_treino=200_000,
        semente=42,
    )
    colunas = ["NU_NOTIFIC", "DT_SIN_PRI_d", "EVOLUCAO", "coorte_hospitalizado",
               "covid_caso", "idade_anos", "CS_SEXO", "UTI", "SUPORT_VEN",
               "linha_deslocada"]
    partes = []
    for lote in pq.ParquetFile(PRATA).iter_batches(columns=colunas, batch_size=250_000):
        df = lote.to_pandas()
        df = df[~df["linha_deslocada"].fillna(False)]
        recorte, _passos = G.cohort_funnel(df, decisoes)  # o MESMO funil do Ouro
        if len(recorte) == 0:
            continue
        partes.append(pd.DataFrame({
            "ano_onset": recorte["DT_SIN_PRI_d"].dt.year.astype("int16").to_numpy(),
            "NU_NOTIFIC": recorte["NU_NOTIFIC"].to_numpy(),
            "y_prata": (recorte["EVOLUCAO"] == "2").astype("int8").to_numpy(),
            "idade_prata": recorte["idade_anos"].astype("float32").to_numpy(),
            "sexo_prata": recorte["CS_SEXO"].map(G.ROTULOS_SEXO).to_numpy(),
            "uti": recorte["UTI"].map(MAPA_UTI).fillna("em_branco").to_numpy(),
            "suport_ven": recorte["SUPORT_VEN"].map(MAPA_SV).fillna("em_branco").to_numpy(),
        }))
    prata = pd.concat(partes, ignore_index=True).sort_values(
        ["ano_onset", "NU_NOTIFIC"], kind="stable", ignore_index=True)
    prata["gold_id"] = np.arange(len(prata), dtype="int32")  # a chave, reconstruída
    print(f"funil do Prata: {len(prata):,} linhas (o Ouro commitado tem 1.282.970)")

    com_uti = amostra.merge(
        prata[["gold_id", "ano_onset", "y_prata", "idade_prata", "sexo_prata",
               "uti", "suport_ven"]].rename(columns={"ano_onset": "ano_prata"}),
        on="gold_id", how="left", validate="1:1")
    confere = {
        "y_obito": int((com_uti["y_obito"] == com_uti["y_prata"]).sum()),
        "idade_anos": int(np.isclose(com_uti["idade_anos"], com_uti["idade_prata"]).sum()),
        "cs_sexo": int((com_uti["cs_sexo"].astype(str) == com_uti["sexo_prata"]).sum()),
        "ano_onset": int((com_uti["ano_onset"] == com_uti["ano_prata"]).sum()),
    }
    assert all(v == len(com_uti) for v in confere.values()), confere
    print(f"realinhamento por gold_id — confere nas {len(com_uti):,} linhas da amostra:")
    for coluna, quantas in confere.items():
        print(f"  {coluna:<12} {quantas:,}")
    com_uti["uti"] = pd.Categorical(com_uti["uti"])
    com_uti["suport_ven"] = pd.Categorical(com_uti["suport_ven"])

funil do Prata: 1,282,970 linhas (o Ouro commitado tem 1.282.970)


realinhamento por gold_id — confere nas 240,290 linhas da amostra:
  y_obito      240,290
  idade_anos   240,290
  cs_sexo      240,290
  ano_onset    240,290


In [9]:
if com_uti is None:
    print("PULADO: exige o Prata local.")
else:
    treino_u = com_uti[com_uti["split"] == "train"]
    for coluna in ("suport_ven", "uti"):
        g_ = treino_u.groupby(coluna, observed=True)["y_obito"].agg(["size", "mean"])
        tabela = pd.DataFrame({
            "n": g_["size"],
            "% dos pacientes": (100 * g_["size"] / len(treino_u)).round(1),
            "letalidade %": (100 * g_["mean"]).round(1),
        })
        print(f"{coluna} — treino ({len(treino_u):,} pacientes)")
        print(tabela.to_string())
        print()
    print("Um campo em que um nível separa 76% de 12% de letalidade não é fator")
    print("de risco: é o desfecho contado de outro jeito.")

suport_ven — treino (200,000 pacientes)
                   n  % dos pacientes  letalidade %
suport_ven                                         
em_branco      17452              8.7          30.2
ignorado        5009              2.5          33.7
invasivo       36000             18.0          76.4
nao            29680             14.8          12.5
nao_invasivo  111859             55.9          21.9

uti — treino (200,000 pacientes)
                n  % dos pacientes  letalidade %
uti                                             
em_branco   14993              7.5          27.9
ignorado     3470              1.7          35.2
nao        114886             57.4          17.8
sim         66651             33.3          55.3

Um campo em que um nível separa 76% de 12% de letalidade não é fator
de risco: é o desfecho contado de outro jeito.


In [10]:
from sklearn.metrics import brier_score_loss

if com_uti is None:
    print("PULADO: exige o Prata local.")
else:
    import xgboost as xgb

    y_u = com_uti["y_obito"].to_numpy()
    e_treino = (com_uti["split"] == "train").to_numpy()
    variantes = {
        "A: as 40": list(M.FEATURES),
        "40+UTI": [*M.FEATURES, "uti"],
        "40+SUPORT_VEN": [*M.FEATURES, "suport_ven"],
        "40+UTI+SUPORT_VEN": [*M.FEATURES, "uti", "suport_ven"],
    }
    modelos_u, p_teste, placar = {}, {}, []
    for nome, feats in variantes.items():
        x_u = com_uti[feats].copy()
        for c in M.BOOLEANAS:
            x_u[c] = x_u[c].astype(bool)
        mdl = xgb.XGBClassifier(**M.XGB_PARAMS)  # os parâmetros adotados
        mdl.fit(x_u[e_treino], y_u[e_treino])
        modelos_u[nome] = mdl
        for s in ("val", "test"):
            m = (com_uti["split"] == s).to_numpy()
            p = mdl.predict_proba(x_u[m])[:, 1]
            if s == "test":
                p_teste[nome] = p
            placar.append({
                "variante": nome, "split": s,
                "auc": round(float(roc_auc_score(y_u[m], p)), 4),
                "brier": round(float(brier_score_loss(y_u[m], p)), 4),
                "previsto médio": round(float(p.mean()), 4),
            })
    print(pd.DataFrame(placar).to_string(index=False))
    print("\nAs linhas `test` acima são leitura EXPLORATÓRIA: o teste de verdade")
    print("foi lido uma vez, no protocolo pré-registrado do SELECTION.md, para o")
    print("modelo adotado. Nada aqui seleciona nada — a seção mede o preço de")
    print("trocar a pergunta, e o preço aparece em qualquer split.")

         variante split    auc  brier  previsto médio
         A: as 40   val 0.7564 0.1356          0.2209
         A: as 40  test 0.7644 0.1308          0.2154
           40+UTI   val 0.7939 0.1284          0.2265
           40+UTI  test 0.8052 0.1232          0.2233
    40+SUPORT_VEN   val 0.8313 0.1141          0.2168
    40+SUPORT_VEN  test 0.8479 0.1087          0.2139
40+UTI+SUPORT_VEN   val 0.8355 0.1135          0.2223
40+UTI+SUPORT_VEN  test 0.8514 0.1084          0.2200

As linhas `test` acima são leitura EXPLORATÓRIA: o teste de verdade
foi lido uma vez, no protocolo pré-registrado do SELECTION.md, para o
modelo adotado. Nada aqui seleciona nada — a seção mede o preço de
trocar a pergunta, e o preço aparece em qualquer split.


In [11]:
if com_uti is None:
    print("PULADO: exige o Prata local.")
else:
    ganho = modelos_u["40+UTI+SUPORT_VEN"].get_booster().get_score(
        importance_type="total_gain")
    total_ganho = sum(ganho.values())
    print("ganho (total_gain) no modelo 40+UTI+SUPORT_VEN — top 5:")
    for nome, valor in sorted(ganho.items(), key=lambda kv: -kv[1])[:5]:
        print(f"  {nome:<22} {100 * valor / total_ganho:5.1f}%")

    y_teste = y_u[(com_uti["split"] == "test").to_numpy()]
    p_a, p_b = p_teste["A: as 40"], p_teste["40+UTI+SUPORT_VEN"]
    rng_boot = np.random.default_rng(42)
    deltas = []
    for _ in range(200):
        idx = rng_boot.integers(0, len(y_teste), len(y_teste))
        if len(np.unique(y_teste[idx])) < 2:
            continue
        deltas.append(roc_auc_score(y_teste[idx], p_b[idx])
                      - roc_auc_score(y_teste[idx], p_a[idx]))
    deltas = np.array(deltas)
    ep = float(deltas.std(ddof=1))
    print("\nbootstrap pareado no teste (200 reamostras, semente 42) — exploratório:")
    print(f"  Δ AUC (42 − 40) = {deltas.mean():+.4f}   EP = {ep:.4f}"
          f"   ({deltas.mean() / ep:.1f} EP)")
    print("\nO ganho é real e enorme. É também a medida de quanto o modelo passou")
    print("a olhar para o futuro do paciente em vez do presente dele.")

ganho (total_gain) no modelo 40+UTI+SUPORT_VEN — top 5:
  suport_ven              49.5%
  idade_anos              23.3%
  uti                      9.0%
  meses_desde_mar2020      3.2%
  regiao                   2.2%



bootstrap pareado no teste (200 reamostras, semente 42) — exploratório:
  Δ AUC (42 − 40) = +0.0868   EP = 0.0038   (22.6 EP)

O ganho é real e enorme. É também a medida de quanto o modelo passou
a olhar para o futuro do paciente em vez do presente dele.


### O que essas tabelas dizem

**O modelo aprendeu uma cópia ruidosa do rótulo.** `suport_ven` sozinho
leva metade do ganho (49,5%, mais que o dobro da idade) e sobe a AUC de
teste de 0,7644 para 0,8479; com `uti` junto, 0,8514 — Δ de +0,0868 no
bootstrap pareado, 22,6 erros-padrão. Nenhum fator de risco de admissão
se comporta assim. O que se comporta assim é o desfecho: quem foi
intubado morreu em 76,4% dos casos, quem não foi, em 12,5%.

**E o nível "não" é uma armadilha dentro da armadilha.** Ele mistura o
paciente leve que nunca precisou de ventilação com o paciente grave a
quem ela não foi oferecida — dois prognósticos opostos no mesmo código,
porque o campo não distingue "não precisou" de "não teve". Ler o efeito
dessa coluna num CP/ICE é ler oferta de leito e conduta médica, não
fisiologia do paciente.

**O critério é o instante da predição** (PROBAST 2.3; TRIPOD+AI 9b): um
preditor tem de existir *quando o modelo seria usado*. Na admissão, nem
`UTI` nem `SUPORT_VEN` existem — a ficha os preenche no encerramento, e
o Guia do SIVEP-Gripe é explícito de que a ficha só fecha depois da
evolução. O placar acima é a demonstração local do que Wynants et al.
encontraram em 2020 nos modelos de COVID: desempenho inflado por
preditores medidos perto demais do desfecho.

**O 0,7644 do curso está no lugar certo.** O 4C Mortality Score (Knight
et al., 2020), construído só com variáveis da admissão, valida em 0,767 —
e o XGBoost comparativo do mesmo trabalho, em 0,779 — para a mesma
pergunta em coorte hospitalar de COVID. As referências completas estão no
[GOLD.md](../GOLD.md). Nosso modelo responde à pergunta do 4C e acerta na mesma faixa;
o modelo de 0,8514 responde a outra.

**Decisão registrada: o curso mantém as 40.** `UTI` e `SUPORT_VEN`
seguem na classe `leakage` ([GOLD.md](../GOLD.md), decisão 3), onde a
justificativa completa — dicionário, metodologia, literatura brasileira
e o confundimento por capacidade instalada — está escrita com as fontes.
Um modelo com essas colunas responderia "quem morreu?"; o do curso
tenta responder "quem vai morrer?".